### 0


In [3]:
import os
import pandas
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_DISABLED"] = "true"

### Squad and Squad2 Evaluate

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer
import numpy as np
from collections import defaultdict
import evaluate

# ----------------------------
# 1. Load dataset
# ----------------------------
squad_v1 = load_dataset("squad")

# ----------------------------
# 2. Preprocess function (shared for all models)
# ----------------------------
max_length = 384
doc_stride = 128

def preprocess_function(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized

# ----------------------------
# 3. Postprocess function
# ----------------------------
def postprocess_qa_predictions(examples, features, predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        valid_answers = []
        context = example["context"]

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                        or end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    text = context[start_char:end_char]
                    score = start_logits[start_index] + end_logits[end_index]
                    valid_answers.append({"score": score, "text": text})

        best_answer = max(valid_answers, key=lambda x: x["score"]) if valid_answers else {"text": ""}
        predictions[example["id"]] = best_answer["text"]

    return predictions

# ----------------------------
# 4. Evaluate multiple models
# ----------------------------
metric = evaluate.load("squad")

model_dirs = [
    "../bert-squad",
    "../distilbert-squad",
    "../albert-squad"
]

results_all = {}

for model_dir in model_dirs:
    print(f"\n🔹 Evaluating {model_dir}...")

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForQuestionAnswering.from_pretrained(model_dir)

    # Tokenize validation set
    tokenized_val = squad_v1["validation"].map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=squad_v1["validation"].column_names
    )

    # Run predictions
    trainer = Trainer(model=model)
    raw_predictions = trainer.predict(tokenized_val)
    start_logits, end_logits = raw_predictions.predictions

    # Postprocess
    final_predictions = postprocess_qa_predictions(
        squad_v1["validation"], tokenized_val, (start_logits, end_logits)
    )

    # Evaluate
    results = metric.compute(
        predictions=[{"id": k, "prediction_text": v} for k, v in final_predictions.items()],
        references=[{"id": ex["id"], "answers": ex["answers"]} for ex in squad_v1["validation"]]
    )

    results_all[model_dir] = results
    print(f"✅ Results for {model_dir}: {results}")

# ----------------------------
# 5. Print comparison
# ----------------------------
print("\n📊 Final Comparison:")
for model_dir, res in results_all.items():
    print(f"{model_dir}: Exact Match = {res['exact_match']:.2f}, F1 = {res['f1']:.2f}")



🔹 Evaluating ../bert-squad...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

e:\Coding\skripsi\.conda\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


  0%|          | 0/1348 [00:00<?, ?it/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: muhammadrizuki (biji). Use `wandb login --relogin` to force relogin


...

  0%|          | 0/1348 [00:00<?, ?it/s]

✅ Results for ../distilbert-squad: {'exact_match': 76.49952696310312, 'f1': 85.25238321387376}

🔹 Evaluating ../albert-squad...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

  0%|          | 0/1351 [00:00<?, ?it/s]

✅ Results for ../albert-squad: {'exact_match': 80.55818353831599, 'f1': 88.73727554431353}

📊 Final Comparison:
../bert-squad: Exact Match = 79.36, F1 = 87.59
../distilbert-squad: Exact Match = 76.50, F1 = 85.25
../albert-squad: Exact Match = 80.56, F1 = 88.74


In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer
import numpy as np
from collections import defaultdict
import evaluate

# ----------------------------
# 1. Load dataset (SQuAD v2)
# ----------------------------
squad_v2 = load_dataset("squad_v2")

# ----------------------------
# 2. Preprocess function
# ----------------------------
max_length = 384
doc_stride = 128

def preprocess_function(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized

# ----------------------------
# 3. Postprocess function (with no-answer handling)
# ----------------------------
def postprocess_qa_predictions(
    examples, features, predictions, n_best_size=20, max_answer_length=30, null_score_diff_threshold=0.0
):
    all_start_logits, all_end_logits = predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        valid_answers = []
        min_null_score = None
        context = example["context"]

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            # null score = start/end logits for CLS token
            cls_index = features[feature_index]["input_ids"].index(tokenizer.cls_token_id)
            feature_null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or feature_null_score < min_null_score:
                min_null_score = feature_null_score

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                        or end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    text = context[start_char:end_char]
                    score = start_logits[start_index] + end_logits[end_index]
                    valid_answers.append({"score": score, "text": text})

        best_answer = {"text": "", "score": min_null_score}
        if valid_answers:
            best_non_null = max(valid_answers, key=lambda x: x["score"])
            if best_non_null["score"] > min_null_score + null_score_diff_threshold:
                best_answer = best_non_null

        predictions[example["id"]] = best_answer["text"]

    return predictions

# ----------------------------
# 4. Evaluate multiple models
# ----------------------------
metric = evaluate.load("squad_v2")

model_dirs = [
    "../bert-squad",
    "../distilbert-squad",
    "../albert-squad"
]

results_all = {}

for model_dir in model_dirs:
    print(f"\n🔹 Evaluating {model_dir}...")

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForQuestionAnswering.from_pretrained(model_dir)

    # Tokenize validation set
    tokenized_val = squad_v2["validation"].map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=squad_v2["validation"].column_names
    )

    # Run predictions
    trainer = Trainer(model=model)
    raw_predictions = trainer.predict(tokenized_val)
    start_logits, end_logits = raw_predictions.predictions

    # Postprocess (with no-answer logic)
    final_predictions = postprocess_qa_predictions(
        squad_v2["validation"], tokenized_val, (start_logits, end_logits)
    )

    # Evaluate
    results = metric.compute(
        predictions=[{"id": k, "prediction_text": v, "no_answer_probability": 0.0} for k, v in final_predictions.items()],
        references=[{"id": ex["id"], "answers": ex["answers"]} for ex in squad_v2["validation"]]
    )

    results_all[model_dir] = results
    print(f"✅ Results for {model_dir}: {results}")

# ----------------------------
# 5. Print comparison
# ----------------------------
print("\n📊 Final Comparison:")
for model_dir, res in results_all.items():
    print(f"{model_dir}: Exact Match = {res['exact']:.2f}, F1 = {res['f1']:.2f}")



🔹 Evaluating ../bert-squad...


  0%|          | 0/1517 [00:00<?, ?it/s]

✅ Results for ../bert-squad: {'exact': 72.25638002189842, 'f1': 76.09186909092948, 'total': 11873, 'HasAns_exact': 72.09851551956815, 'HasAns_f1': 79.78049286717376, 'HasAns_total': 5928, 'NoAns_exact': 72.41379310344827, 'NoAns_f1': 72.41379310344827, 'NoAns_total': 5945, 'best_exact': 72.25638002189842, 'best_exact_thresh': 0.0, 'best_f1': 76.09186909092945, 'best_f1_thresh': 0.0}

🔹 Evaluating ../distilbert-squad...


Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

  0%|          | 0/1517 [00:00<?, ?it/s]

✅ Results for ../distilbert-squad: {'exact': 63.52227743619978, 'f1': 67.27332771815476, 'total': 11873, 'HasAns_exact': 68.28609986504723, 'HasAns_f1': 75.79895748948188, 'HasAns_total': 5928, 'NoAns_exact': 58.772077375946175, 'NoAns_f1': 58.772077375946175, 'NoAns_total': 5945, 'best_exact': 63.52227743619978, 'best_exact_thresh': 0.0, 'best_f1': 67.27332771815495, 'best_f1_thresh': 0.0}

🔹 Evaluating ../albert-squad...


Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

  0%|          | 0/1522 [00:00<?, ?it/s]

✅ Results for ../albert-squad: {'exact': 76.92242904068054, 'f1': 80.77489116342453, 'total': 11873, 'HasAns_exact': 72.72267206477733, 'HasAns_f1': 80.43864419422061, 'HasAns_total': 5928, 'NoAns_exact': 81.11017661900758, 'NoAns_f1': 81.11017661900758, 'NoAns_total': 5945, 'best_exact': 76.92242904068054, 'best_exact_thresh': 0.0, 'best_f1': 80.77489116342443, 'best_f1_thresh': 0.0}

📊 Final Comparison:
../bert-squad: Exact Match = 72.26, F1 = 76.09
../distilbert-squad: Exact Match = 63.52, F1 = 67.27
../albert-squad: Exact Match = 76.92, F1 = 80.77


### 1.5

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer
import numpy as np
from collections import defaultdict
import evaluate

# ----------------------------
# 1. Load dataset
# ----------------------------
squad_v1 = load_dataset("squad")

# ----------------------------
# 2. Preprocess function (shared for all models)
# ----------------------------
max_length = 384
doc_stride = 128

def preprocess_function(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized

# ----------------------------
# 3. Postprocess function
# ----------------------------
def postprocess_qa_predictions(examples, features, predictions, n_best_size=20, max_answer_length=30):
    all_start_logits, all_end_logits = predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        valid_answers = []
        context = example["context"]

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                        or end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    text = context[start_char:end_char]
                    score = start_logits[start_index] + end_logits[end_index]
                    valid_answers.append({"score": score, "text": text})

        best_answer = max(valid_answers, key=lambda x: x["score"]) if valid_answers else {"text": ""}
        predictions[example["id"]] = best_answer["text"]

    return predictions

# ----------------------------
# 4. Evaluate multiple models
# ----------------------------
metric = evaluate.load("squad")

model_dirs = [
    "google-bert/bert-base-uncased",
    "distilbert/distilbert-base-uncased",
    "albert/albert-base-v2"
]

results_all = {}

for model_dir in model_dirs:
    print(f"\n🔹 Evaluating {model_dir}...")

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForQuestionAnswering.from_pretrained(model_dir)

    # Tokenize validation set
    tokenized_val = squad_v1["validation"].map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=squad_v1["validation"].column_names
    )

    # Run predictions
    trainer = Trainer(model=model)
    raw_predictions = trainer.predict(tokenized_val)
    start_logits, end_logits = raw_predictions.predictions

    # Postprocess
    final_predictions = postprocess_qa_predictions(
        squad_v1["validation"], tokenized_val, (start_logits, end_logits)
    )

    # Evaluate
    results = metric.compute(
        predictions=[{"id": k, "prediction_text": v} for k, v in final_predictions.items()],
        references=[{"id": ex["id"], "answers": ex["answers"]} for ex in squad_v1["validation"]]
    )

    results_all[model_dir] = results
    print(f"✅ Results for {model_dir}: {results}")

# ----------------------------
# 5. Print comparison
# ----------------------------
print("\n📊 Final Comparison:")
for model_dir, res in results_all.items():
    print(f"{model_dir}: Exact Match = {res['exact_match']:.2f}, F1 = {res['f1']:.2f}")


In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer
import numpy as np
from collections import defaultdict
import evaluate

# ----------------------------
# 1. Load dataset (SQuAD v2)
# ----------------------------
squad_v2 = load_dataset("squad_v2")

# ----------------------------
# 2. Preprocess function
# ----------------------------
max_length = 384
doc_stride = 128

def preprocess_function(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized

# ----------------------------
# 3. Postprocess function (with no-answer handling)
# ----------------------------
def postprocess_qa_predictions(
    examples, features, predictions, n_best_size=20, max_answer_length=30, null_score_diff_threshold=0.0
):
    all_start_logits, all_end_logits = predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        valid_answers = []
        min_null_score = None
        context = example["context"]

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            # null score = start/end logits for CLS token
            cls_index = features[feature_index]["input_ids"].index(tokenizer.cls_token_id)
            feature_null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or feature_null_score < min_null_score:
                min_null_score = feature_null_score

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                        or end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    text = context[start_char:end_char]
                    score = start_logits[start_index] + end_logits[end_index]
                    valid_answers.append({"score": score, "text": text})

        best_answer = {"text": "", "score": min_null_score}
        if valid_answers:
            best_non_null = max(valid_answers, key=lambda x: x["score"])
            if best_non_null["score"] > min_null_score + null_score_diff_threshold:
                best_answer = best_non_null

        predictions[example["id"]] = best_answer["text"]

    return predictions

# ----------------------------
# 4. Evaluate multiple models
# ----------------------------
metric = evaluate.load("squad_v2")

model_dirs = [
    "rizukim/distilbert-base-uncased-finetuned-squad-v2"
]

results_all = {}

for model_dir in model_dirs:
    print(f"\n🔹 Evaluating {model_dir}...")

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForQuestionAnswering.from_pretrained(model_dir)

    # Tokenize validation set
    tokenized_val = squad_v2["validation"].map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=squad_v2["validation"].column_names
    )

    # Run predictions
    trainer = Trainer(model=model)
    raw_predictions = trainer.predict(tokenized_val)
    start_logits, end_logits = raw_predictions.predictions

    # Postprocess (with no-answer logic)
    final_predictions = postprocess_qa_predictions(
        squad_v2["validation"], tokenized_val, (start_logits, end_logits)
    )

    # Evaluate
    results = metric.compute(
        predictions=[{"id": k, "prediction_text": v, "no_answer_probability": 0.0} for k, v in final_predictions.items()],
        references=[{"id": ex["id"], "answers": ex["answers"]} for ex in squad_v2["validation"]]
    )

    results_all[model_dir] = results
    print(f"✅ Results for {model_dir}: {results}")

# ----------------------------
# 5. Print comparison
# ----------------------------
print("\n📊 Final Comparison:")
for model_dir, res in results_all.items():
    print(f"{model_dir}: Exact Match = {res['exact']:.2f}, F1 = {res['f1']:.2f}")



🔹 Evaluating rizukim/distilbert-base-uncased-finetuned-squad-v2...


tokenizer_config.json: 0.00B [00:00, ?B/s]

e:\Coding\skripsi\.conda\lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rogg5\.cache\huggingface\hub\models--rizukim--distilbert-base-uncased-finetuned-squad-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/11873 [00:00<?, ? examples/s]

e:\Coding\skripsi\.conda\lib\site-packages\transformers\models\distilbert\modeling_distilbert.py:403: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


  0%|          | 0/1517 [00:00<?, ?it/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: muhammadrizuki (biji). Use `wandb login --relogin` to force relogin


✅ Results for rizukim/distilbert-base-uncased-finetuned-squad-v2: {'exact': 64.65088857070664, 'f1': 68.09760127379433, 'total': 11873, 'HasAns_exact': 67.10526315789474, 'HasAns_f1': 74.0085728616327, 'HasAns_total': 5928, 'NoAns_exact': 62.20353238015139, 'NoAns_f1': 62.20353238015139, 'NoAns_total': 5945, 'best_exact': 64.65088857070664, 'best_exact_thresh': 0.0, 'best_f1': 68.09760127379452, 'best_f1_thresh': 0.0}

📊 Final Comparison:
rizukim/distilbert-base-uncased-finetuned-squad-v2: Exact Match = 64.65, F1 = 68.10


### Train (Not Used)

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, Trainer, TrainingArguments, TrainerCallback
from datasets import load_dataset
import evaluate
import numpy as np
import pandas as pd

# -------------------------------
# 1. Load dataset
# -------------------------------
dataset = load_dataset("rajpurkar/squad_v2")

# -------------------------------
# 2. Load model & tokenizer
# -------------------------------
model_name = "bert-base-uncased"
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# -------------------------------
# 3. Preprocessing function
# -------------------------------
max_length = 384
doc_stride = 128

def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]

    # Tokenize with sliding window (doc_stride helps with long contexts)
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    # Map examples to features
    sample_mapping = inputs.pop("overflow_to_sample_mapping")
    offset_mapping = inputs.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = inputs["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = inputs.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:  # No answer case
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Find token start and end
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1

            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            # If answer is outside the span, use CLS token
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                start_positions.append(cls_index)
                end_positions.append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                start_positions.append(token_start_index - 1)

                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                end_positions.append(token_end_index + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

# -------------------------------
# 4. Metrics (official squad_v2 metric)
# -------------------------------
metric = evaluate.load("squad_v2")

def compute_metrics(p):
    start_logits, end_logits = p.predictions
    n_best = 20
    max_answer_length = 30

    examples = dataset["validation"]
    features = tokenized_datasets["validation"]

    predictions = []
    references = []

    for i, example in enumerate(examples):
        start_idx = int(np.argmax(start_logits[i]))
        end_idx = int(np.argmax(end_logits[i]))

        pred_ids = features[i]["input_ids"][start_idx:end_idx+1]
        pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True)

        predictions.append({"id": example["id"], "prediction_text": pred_text, "no_answer_probability": 0.0})
        references.append({"id": example["id"], "answers": example["answers"]})

    return metric.compute(predictions=predictions, references=references)

# -------------------------------
# 5. Custom callback (logging)
# -------------------------------
class PrintMetricsCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if kwargs.get("metrics"):
            metrics = kwargs["metrics"]
            epoch = state.epoch
            table = pd.DataFrame(
                [[epoch, metrics.get("loss", "N/A"), metrics.get("eval_loss", "N/A"),
                  metrics.get("eval_f1", "N/A"), metrics.get("eval_exact", "N/A")]],
                columns=["Epoch", "Train Loss", "Val Loss", "F1", "Exact Match"]
            )
            print("\n" + table.to_string(index=False) + "\n")

# -------------------------------
# 6. Training configuration
# -------------------------------
training_args = TrainingArguments(
    output_dir="./bert-squad-v2",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=200,
    save_steps=500,
    warmup_steps=500,
    fp16=True,
    report_to="none"
)

# -------------------------------
# 7. Trainer
# -------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.add_callback(PrintMetricsCallback())

# -------------------------------
# 8. Train & Save
# -------------------------------
trainer.train()
trainer.save_model("./bert-squad-v2-model")


Using the latest cached version of the dataset since rajpurkar/squad_v2 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'squad_v2' at C:\Users\rogg5\.cache\huggingface\datasets\rajpurkar___squad_v2\squad_v2\0.0.0\3ffb306f725f7d2ce8394bc1873b24868140c412 (last modified on Tue Sep  2 13:26:34 2025).
Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\rogg5\AppData\Local\Temp\ipykernel_23076\2211783277.py:156: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


  0%|          | 0/16470 [00:00<?, ?it/s]

{'loss': 4.6417, 'grad_norm': 9.366747856140137, 'learning_rate': 1.1880000000000001e-05, 'epoch': 0.02}
{'loss': 2.6143, 'grad_norm': 11.66221809387207, 'learning_rate': 2.3880000000000002e-05, 'epoch': 0.05}
{'loss': 2.0517, 'grad_norm': 19.291433334350586, 'learning_rate': 2.9815904821540387e-05, 'epoch': 0.07}
{'loss': 1.7893, 'grad_norm': 16.856035232543945, 'learning_rate': 2.944020037570445e-05, 'epoch': 0.1}
{'loss': 1.6722, 'grad_norm': 12.949929237365723, 'learning_rate': 2.906637445209768e-05, 'epoch': 0.12}
{'loss': 1.5689, 'grad_norm': 18.648624420166016, 'learning_rate': 2.869067000626174e-05, 'epoch': 0.15}
{'loss': 1.5611, 'grad_norm': 21.212413787841797, 'learning_rate': 2.83149655604258e-05, 'epoch': 0.17}
{'loss': 1.4822, 'grad_norm': 16.326210021972656, 'learning_rate': 2.7939261114589858e-05, 'epoch': 0.19}
{'loss': 1.4091, 'grad_norm': 17.072717666625977, 'learning_rate': 2.7563556668753916e-05, 'epoch': 0.22}
{'loss': 1.4012, 'grad_norm': 8.900731086730957, 'lear

  0%|          | 0/759 [00:00<?, ?it/s]

{'eval_loss': 1.0964275598526, 'eval_exact': 22.06687442095511, 'eval_f1': 22.378743099678307, 'eval_total': 11873, 'eval_HasAns_exact': 0.43859649122807015, 'eval_HasAns_f1': 1.0632282089204714, 'eval_HasAns_total': 5928, 'eval_NoAns_exact': 43.63330529857023, 'eval_NoAns_f1': 43.63330529857023, 'eval_NoAns_total': 5945, 'eval_best_exact': 50.10528088941295, 'eval_best_exact_thresh': 0.0, 'eval_best_f1': 50.10902420992541, 'eval_best_f1_thresh': 0.0, 'eval_runtime': 187.7829, 'eval_samples_per_second': 64.617, 'eval_steps_per_second': 4.042, 'epoch': 1.0}
{'loss': 0.789, 'grad_norm': 12.219501495361328, 'learning_rate': 1.5169067000626176e-05, 'epoch': 1.02}
{'loss': 0.7529, 'grad_norm': 11.592742919921875, 'learning_rate': 1.4793362554790232e-05, 'epoch': 1.04}
{'loss': 0.7548, 'grad_norm': 12.44871711730957, 'learning_rate': 1.441765810895429e-05, 'epoch': 1.07}
{'loss': 0.7373, 'grad_norm': 19.05113410949707, 'learning_rate': 1.4041953663118347e-05, 'epoch': 1.09}
{'loss': 0.7985, 

  0%|          | 0/759 [00:00<?, ?it/s]

{'eval_loss': 1.1811312437057495, 'eval_exact': 21.056177882590752, 'eval_f1': 21.385927393826936, 'eval_total': 11873, 'eval_HasAns_exact': 0.47233468286099867, 'eval_HasAns_f1': 1.1327793432704654, 'eval_HasAns_total': 5928, 'eval_NoAns_exact': 41.581160639192596, 'eval_NoAns_f1': 41.581160639192596, 'eval_NoAns_total': 5945, 'eval_best_exact': 50.11370336056599, 'eval_best_exact_thresh': 0.0, 'eval_best_f1': 50.11370336056599, 'eval_best_f1_thresh': 0.0, 'eval_runtime': 190.4902, 'eval_samples_per_second': 63.699, 'eval_steps_per_second': 3.984, 'epoch': 2.0}
{'train_runtime': 18048.8969, 'train_samples_per_second': 14.6, 'train_steps_per_second': 0.913, 'train_loss': 1.0396282881625, 'epoch': 2.0}


### Weigth in Models

In [4]:
from transformers import AutoModel

model_dirs = [
    "../bert-squad",
    "../distilbert-squad",
    "../albert-squad"
]

model = AutoModel.from_pretrained(model_dirs[0])

for name, param in model.named_parameters():
    print(name, param.data)
    break   # remove 'break' to show ALL weights


Some weights of BertModel were not initialized from the model checkpoint at ../bert-squad and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


embeddings.word_embeddings.weight tensor([[-0.0101, -0.0613, -0.0264,  ..., -0.0198, -0.0370, -0.0097],
        [-0.0117, -0.0598, -0.0322,  ..., -0.0167, -0.0399, -0.0106],
        [-0.0197, -0.0624, -0.0325,  ..., -0.0164, -0.0418, -0.0032],
        ...,
        [-0.0217, -0.0554, -0.0134,  ..., -0.0043, -0.0151, -0.0248],
        [-0.0460, -0.0562, -0.0019,  ...,  0.0156, -0.0138, -0.0094],
        [ 0.0015, -0.0817, -0.0159,  ..., -0.0081, -0.0473,  0.0749]])


In [5]:
for name, param in model.named_parameters(): print(name, param.data.mean().item(), param.data.std().item())

embeddings.word_embeddings.weight -0.0279032401740551 0.0425211600959301
embeddings.position_embeddings.weight -3.825477324426174e-05 0.016075942665338516
embeddings.token_type_embeddings.weight -0.0009030437213368714 0.033443741500377655
embeddings.LayerNorm.weight 0.8491108417510986 0.13448312878608704
embeddings.LayerNorm.bias -0.019674809649586678 0.0694945827126503
encoder.layer.0.attention.self.query.weight 2.9962506232550368e-05 0.04298923909664154
encoder.layer.0.attention.self.query.bias 0.00638827309012413 0.28108611702919006
encoder.layer.0.attention.self.key.weight -1.889729446702404e-06 0.042662739753723145
encoder.layer.0.attention.self.key.bias -3.2689909858163446e-05 0.004586847499012947
encoder.layer.0.attention.self.value.weight -4.0625262045068666e-05 0.02870893105864525
encoder.layer.0.attention.self.value.bias 0.0018777014920488 0.04032925143837929
encoder.layer.0.attention.output.dense.weight -1.9924984371755272e-05 0.028479116037487984
encoder.layer.0.attention.o

In [2]:
import torch

for name, param in model.named_parameters():
    data = param.data
    mean = data.mean().item()
    std  = data.std().item()
    mn   = data.min().item()
    mx   = data.max().item()
    finite_all = torch.isfinite(data).all().item()

    if not finite_all:
        print("NON-FINITE:", name, "mean", mean, "std", std)
    if mean < -1:
        print("MEAN < -1:", name, "mean", mean, "std", std, "min", mn, "max", mx)
    if std == 0:
        print("ZERO STD:", name, "mean", mean, "min", mn, "max", mx)


ZERO STD: pooler.dense.bias mean 0.0 min 0.0 max 0.0


In [9]:
from transformers import AutoModel

model = AutoModel.from_pretrained("google-bert/bert-base-uncased")

for name, param in model.named_parameters():
    print(name, param.data.mean().item(), param.data.std().item())


embeddings.word_embeddings.weight -0.02802450768649578 0.04266705736517906
embeddings.position_embeddings.weight -3.873982495861128e-05 0.016068821772933006
embeddings.token_type_embeddings.weight -0.0009192298166453838 0.033554624766111374
embeddings.LayerNorm.weight 0.8493084907531738 0.13398823142051697
embeddings.LayerNorm.bias -0.019596023485064507 0.06929602473974228
encoder.layer.0.attention.self.query.weight 3.558379830792546e-05 0.04311944171786308
encoder.layer.0.attention.self.query.bias 0.00633581355214119 0.2808695137500763
encoder.layer.0.attention.self.key.weight 4.959372745361179e-06 0.042788852006196976
encoder.layer.0.attention.self.key.bias -0.0003075960266869515 0.0031877632718533278
encoder.layer.0.attention.self.value.weight -3.300314347143285e-05 0.028810927644371986
encoder.layer.0.attention.self.value.bias 0.0018207309767603874 0.040716055780649185
encoder.layer.0.attention.output.dense.weight -2.0735598809551448e-05 0.028576454147696495
encoder.layer.0.attenti

### Next Word Guessing

In [20]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="bert-base-uncased")
result = fill_mask("The capital city of France is [MASK].")

print("Sentence: The capital city of France is [MASK].")
print("Predictions [MASK]")
for r in result:
    print(r["token_str"], "-", r["score"])

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Sentence: The capital city of France is [MASK].
Predictions [MASK]
paris - 0.34645143151283264
lille - 0.09251449257135391
lyon - 0.0850038155913353
marseille - 0.06234076991677284
toulouse - 0.040497224777936935


### Exploring Train Data CSV

In [88]:
# import csv
# with open('../distilbert-squad/training_results.csv', mode ='r')as file:
#   csvFile = csv.reader(file)
#   for lines in csvFile:
#         print(lines)
import pandas
import math

index = 0

df = pandas.read_csv('../bert-squad/training_results_end.csv')

for i in df['eval_loss']:
    index += 1
    if i is not None and not math.isnan(i):
        print(df.iloc[index-1])
        # print(df.iloc[index])



loss                                NaN
grad_norm                           NaN
learning_rate                       NaN
epoch                          1.000000
step                        8145.000000
eval_loss                      0.987058
eval_accuracy                  0.617283
eval_f1                        0.546902
eval_exact_match               0.690937
eval_runtime                  50.813100
eval_samples_per_second      233.660000
eval_steps_per_second         29.225000
train_runtime                       NaN
train_samples_per_second            NaN
train_steps_per_second              NaN
total_flos                          NaN
train_loss                          NaN
Name: 81, dtype: float64
loss                                 NaN
grad_norm                            NaN
learning_rate                        NaN
epoch                           2.000000
step                        16290.000000
eval_loss                       1.127349
eval_accuracy                   0.620062
eval_f1 

In [90]:
df.iloc[-3]

loss                        2.858000e-01
grad_norm                   2.057391e+01
learning_rate               2.137974e-08
epoch                       4.996931e+00
step                        4.070000e+04
eval_loss                            NaN
eval_accuracy                        NaN
eval_f1                              NaN
eval_exact_match                     NaN
eval_runtime                         NaN
eval_samples_per_second              NaN
eval_steps_per_second                NaN
train_runtime                        NaN
train_samples_per_second             NaN
train_steps_per_second               NaN
total_flos                           NaN
train_loss                           NaN
Name: 410, dtype: float64

In [24]:
df

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_accuracy,eval_f1,eval_exact_match,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,5.4901,4.843753,3.920000e-06,0.012277,100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.2964,8.389332,7.920000e-06,0.024555,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3.2965,7.561268,1.192000e-05,0.036832,300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3.0661,14.455779,1.592000e-05,0.049110,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2.7447,15.455652,1.988000e-05,0.061387,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,0.4782,16.161392,1.208204e-07,4.972376,40500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
409,0.4760,20.889374,7.110006e-08,4.984653,40600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
410,0.4670,19.748877,2.187694e-08,4.996931,40700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
411,NaN,NaN,NaN,5.000000,40725,1.858591,0.524888,0.498326,0.598627,326.2632,36.391,4.552,NaN,NaN,NaN,NaN,NaN


### Comparing distilbert

In [9]:
# -----------------------------------------------------------
# 📌 Install Transformers
# -----------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

# -----------------------------------------------------------
# 📌 Choose your two QA models here
# -----------------------------------------------------------
model_name_1 = "E:/Coding/skripsi/oktober14/distilbert-squad"
model_name_2 = "rizukim/distilbert-base-uncased-finetuned-squad-v2"

# -----------------------------------------------------------
# 📌 Load the models + tokenizers
# -----------------------------------------------------------
tokenizer1 = AutoTokenizer.from_pretrained(model_name_1)
model1 = AutoModelForQuestionAnswering.from_pretrained(model_name_1)

tokenizer2 = AutoTokenizer.from_pretrained(model_name_2)
model2 = AutoModelForQuestionAnswering.from_pretrained(model_name_2)

qa1 = pipeline("question-answering", model=model1, tokenizer=tokenizer1)
qa2 = pipeline("question-answering", model=model2, tokenizer=tokenizer2)

# -----------------------------------------------------------
# 📌 Input: Same context & question for both models
# -----------------------------------------------------------
context = """
The Transformers library was created by Hugging Face and 
provides thousands of pre-trained models to perform tasks 
on texts such as classification, information extraction, 
question answering, and text generation.
"""

questions = [
    "Who created the Transformers library?",
    "What does the Transformers library provide?",
    "What kind of tasks can the Transformers models perform?"
]

# -----------------------------------------------------------
# 📌 Run QA pipeline for each question
# -----------------------------------------------------------

rows = []

for i, q in enumerate(questions, start=1):

    ans1 = qa1(question=q, context=context)
    ans2 = qa2(question=q, context=context)

    rows.append({
        "model": model_name_1.split('/')[-1],
        "question": q,
        "answer": ans1["answer"],
        "score": ans1["score"]
    })

    rows.append({
        "model": model_name_2,
        "question": q,
        "answer": ans2["answer"],
        "score": ans2["score"]
    })

pandas.set_option('display.max_colwidth', None)   # do NOT cut long text
pandas.set_option('display.max_columns', None)   # show all columns
pandas.set_option('display.width', 2000)
df = pandas.DataFrame(rows).style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

df

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


,model,question,answer,score
0,distilbert-squad,Who created the Transformers library?,Hugging Face,0.946947
1,rizukim/distilbert-base-uncased-finetuned-squad-v2,Who created the Transformers library?,Hugging Face,0.988969
2,distilbert-squad,What does the Transformers library provide?,thousands of pre-trained models to perform tasks on texts,0.380340
3,rizukim/distilbert-base-uncased-finetuned-squad-v2,What does the Transformers library provide?,thousands of pre-trained models,0.398643
4,distilbert-squad,What kind of tasks can the Transformers models perform?,"texts such as classification, information extraction, question answering, and text generation",0.242396
5,rizukim/distilbert-base-uncased-finetuned-squad-v2,What kind of tasks can the Transformers models perform?,"texts such as classification, information extraction, question answering, and text generation",0.108864


### Comparing BERT

In [8]:
# -----------------------------------------------------------
# 📌 Install Transformers
# -----------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

# -----------------------------------------------------------
# 📌 Choose your two QA models here
# -----------------------------------------------------------
model_name_1 = "E:/Coding/skripsi/oktober14/bert-squad"
model_name_2 = "deepset/bert-base-uncased-squad2"

# -----------------------------------------------------------
# 📌 Load the models + tokenizers
# -----------------------------------------------------------
tokenizer1 = AutoTokenizer.from_pretrained(model_name_1)
model1 = AutoModelForQuestionAnswering.from_pretrained(model_name_1)

tokenizer2 = AutoTokenizer.from_pretrained(model_name_2)
model2 = AutoModelForQuestionAnswering.from_pretrained(model_name_2)

qa1 = pipeline("question-answering", model=model1, tokenizer=tokenizer1)
qa2 = pipeline("question-answering", model=model2, tokenizer=tokenizer2)

# -----------------------------------------------------------
# 📌 Input: Same context & question for both models
# -----------------------------------------------------------
context = """
The Transformers library was created by Hugging Face and 
provides thousands of pre-trained models to perform tasks 
on texts such as classification, information extraction, 
question answering, and text generation.
"""

questions = [
    "Who created the Transformers library?",
    "What does the Transformers library provide?",
    "What kind of tasks can the Transformers models perform?"
]

# -----------------------------------------------------------
# 📌 Run QA pipeline for each question
# -----------------------------------------------------------

rows = []

for i, q in enumerate(questions, start=1):

    ans1 = qa1(question=q, context=context)
    ans2 = qa2(question=q, context=context)

    rows.append({
        "model": model_name_1.split('/')[-1],
        "question": q,
        "answer": ans1["answer"],
        "score": ans1["score"]
    })

    rows.append({
        "model": model_name_2,
        "question": q,
        "answer": ans2["answer"],
        "score": ans2["score"]
    })
    
pandas.set_option('display.max_colwidth', None)   # do NOT cut long text
pandas.set_option('display.max_columns', None)   # show all columns
pandas.set_option('display.width', 2000)
df = pandas.DataFrame(rows).style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

df

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


,model,question,answer,score
0,bert-squad,Who created the Transformers library?,Hugging Face,0.999772
1,deepset/bert-base-uncased-squad2,Who created the Transformers library?,Hugging Face,0.996191
2,bert-squad,What does the Transformers library provide?,thousands of pre-trained models,0.911247
3,deepset/bert-base-uncased-squad2,What does the Transformers library provide?,thousands of pre-trained models,0.844383
4,bert-squad,What kind of tasks can the Transformers models perform?,"classification, information extraction, question answering, and text generation",0.638348
5,deepset/bert-base-uncased-squad2,What kind of tasks can the Transformers models perform?,"classification, information extraction, question answering, and text generation",0.486195


### Compare ALBERT

In [10]:
# -----------------------------------------------------------
# 📌 Install Transformers
# -----------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

# -----------------------------------------------------------
# 📌 Choose your two QA models here
# -----------------------------------------------------------
model_name_1 = "E:/Coding/skripsi/oktober14/albert-squad"
model_name_2 = "twmkn9/albert-base-v2-squad2"

# -----------------------------------------------------------
# 📌 Load the models + tokenizers
# -----------------------------------------------------------
tokenizer1 = AutoTokenizer.from_pretrained(model_name_1)
model1 = AutoModelForQuestionAnswering.from_pretrained(model_name_1)

tokenizer2 = AutoTokenizer.from_pretrained(model_name_2)
model2 = AutoModelForQuestionAnswering.from_pretrained(model_name_2)

qa1 = pipeline("question-answering", model=model1, tokenizer=tokenizer1)
qa2 = pipeline("question-answering", model=model2, tokenizer=tokenizer2)

# -----------------------------------------------------------
# 📌 Input: Same context & question for both models
# -----------------------------------------------------------
context = """
The Transformers library was created by Hugging Face and 
provides thousands of pre-trained models to perform tasks 
on texts such as classification, information extraction, 
question answering, and text generation.
"""

questions = [
    "Who created the Transformers library?",
    "What does the Transformers library provide?",
    "What kind of tasks can the Transformers models perform?"
]

# -----------------------------------------------------------
# 📌 Run QA pipeline for each question
# -----------------------------------------------------------

rows = []

for i, q in enumerate(questions, start=1):

    ans1 = qa1(question=q, context=context)
    ans2 = qa2(question=q, context=context)

    rows.append({
        "model": model_name_1.split('/')[-1],
        "question": q,
        "answer": ans1["answer"],
        "score": ans1["score"]
    })

    rows.append({
        "model": model_name_2,
        "question": q,
        "answer": ans2["answer"],
        "score": ans2["score"]
    })

pandas.set_option('display.max_colwidth', None)   # do NOT cut long text
pandas.set_option('display.max_columns', None)   # show all columns
pandas.set_option('display.width', 2000)
df = pandas.DataFrame(rows).style.set_properties(**{'text-align': 'left'}).set_table_styles(
    [{'selector': 'th', 'props': [('text-align', 'left')]}]
)

df

Some weights of the model checkpoint at twmkn9/albert-base-v2-squad2 were not used when initializing AlbertForQuestionAnswering: ['albert.pooler.bias', 'albert.pooler.weight']
- This IS expected if you are initializing AlbertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing AlbertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


,model,question,answer,score
0,albert-squad,Who created the Transformers library?,Hugging Face,0.999105
1,twmkn9/albert-base-v2-squad2,Who created the Transformers library?,Hugging Face,0.992993
2,albert-squad,What does the Transformers library provide?,provides thousands of pre-trained models to perform tasks on texts,0.110878
3,twmkn9/albert-base-v2-squad2,What does the Transformers library provide?,thousands of pre-trained models,0.571872
4,albert-squad,What kind of tasks can the Transformers models perform?,"classification, information extraction, question answering, and text generation.",0.432775
5,twmkn9/albert-base-v2-squad2,What kind of tasks can the Transformers models perform?,pre-trained,0.380696


### Scoring answer

In [81]:
tokenizer = AutoTokenizer.from_pretrained("E:/Coding/skripsi/oktober14/bert-squad")
model = AutoModelForQuestionAnswering.from_pretrained("E:/Coding/skripsi/oktober14/bert-squad")

qa = pipeline("question-answering", model=model, tokenizer=tokenizer, top_k=5)

answers = qa(question=questions[2], context=context)

df = pandas.DataFrame(answers)

df


Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


,score,start,end,answer
0,0.638348,135,215,"classification, information extraction, \nques..."
1,0.039818,121,215,"texts such as classification, information extr..."
2,0.039545,135,216,"classification, information extraction, \nques..."
3,0.021610,151,215,"information extraction, \nquestion answering, ..."
4,0.017015,135,149,classification


### Validate Checkpoint

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer
import numpy as np
from collections import defaultdict
import evaluate

# ----------------------------
# 1. Load dataset (SQuAD v2)
# ----------------------------
squad_v2 = load_dataset("squad_v2")

# ----------------------------
# 2. Preprocess function
# ----------------------------
max_length = 384
doc_stride = 128

def preprocess_function(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_idx])

        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized

# ----------------------------
# 3. Postprocess function (with no-answer handling)
# ----------------------------
def postprocess_qa_predictions(
    examples, features, predictions, n_best_size=20, max_answer_length=30, null_score_diff_threshold=0.0
):
    all_start_logits, all_end_logits = predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        feature_indices = features_per_example[example_index]
        valid_answers = []
        min_null_score = None
        context = example["context"]

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            # null score = start/end logits for CLS token
            cls_index = features[feature_index]["input_ids"].index(tokenizer.cls_token_id)
            feature_null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or feature_null_score < min_null_score:
                min_null_score = feature_null_score

            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1: -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1: -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                        or end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    text = context[start_char:end_char]
                    score = start_logits[start_index] + end_logits[end_index]
                    valid_answers.append({"score": score, "text": text})

        best_answer = {"text": "", "score": min_null_score}
        if valid_answers:
            best_non_null = max(valid_answers, key=lambda x: x["score"])
            if best_non_null["score"] > min_null_score + null_score_diff_threshold:
                best_answer = best_non_null

        predictions[example["id"]] = best_answer["text"]

    return predictions

# ----------------------------
# 4. Evaluate multiple models
# ----------------------------
metric = evaluate.load("squad_v2")

model_dirs = [
    "E:/Coding/skripsi/rev/bert-base-uncased-finetuned-squad-v2/checkpoint-8235",
    "E:/Coding/skripsi/rev/bert-base-uncased-finetuned-squad-v2/checkpoint-16470",
    "E:/Coding/skripsi/rev/bert-base-uncased-finetuned-squad-v2/checkpoint-24705",
]

results_all = {}

for model_dir in model_dirs:
    print(f"\n🔹 Evaluating {model_dir}...")

    # Load tokenizer & model
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForQuestionAnswering.from_pretrained(model_dir)

    # Tokenize validation set
    tokenized_val = squad_v2["validation"].map(
        lambda x: preprocess_function(x, tokenizer),
        batched=True,
        remove_columns=squad_v2["validation"].column_names
    )

    # Run predictions
    trainer = Trainer(model=model)
    raw_predictions = trainer.predict(tokenized_val)
    start_logits, end_logits = raw_predictions.predictions

    # Postprocess (with no-answer logic)
    final_predictions = postprocess_qa_predictions(
        squad_v2["validation"], tokenized_val, (start_logits, end_logits)
    )

    # Evaluate
    results = metric.compute(
        predictions=[{"id": k, "prediction_text": v, "no_answer_probability": 0.0} for k, v in final_predictions.items()],
        references=[{"id": ex["id"], "answers": ex["answers"]} for ex in squad_v2["validation"]]
    )

    results_all[model_dir] = results
    print(f"✅ Results for {model_dir.split('/')[-1]}: {results}")

# ----------------------------
# 5. Print comparison
# ----------------------------
print("\n📊 Final Comparison:")
for model_dir, res in results_all.items():
    print(f"{model_dir.split('/')[-1]}: Exact Match = {res['exact']:.2f}, F1 = {res['f1']:.2f}")



📊 Final Comparison:
checkpoint-8235: Exact Match = 70.78, F1 = 73.90
checkpoint-16470: Exact Match = 72.74, F1 = 76.19
checkpoint-24705: Exact Match = 72.93, F1 = 76.37
